In [15]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os, time, copy
# import matplotlib
# matplotlib.use('TkAgg')

os.environ['TZ'] = 'Asia/Ho_Chi_Minh' # set new timezone
# time.tzset()

In [16]:
# # load csv that contain groundtruth
# import pandas as pd
# df = pd.read_csv('/home/dev1/Downloads/UHCTD Comprehensive Dataset For Camera Tampering Detection/Camera B/Testing video/Day 3/annotations.csv')
# df.head()

In [17]:
# df.loc[df['is tampered'] == 2]  # find tamper in csv file: 1: covered, 2: blur, 3: moved

### Config

In [18]:
cap = cv2.VideoCapture(0)  # built-in Windows camera
fgbg = cv2.createBackgroundSubtractorMOG2(detectShadows=False)
fgbg.setNMixtures(4)
fgbg.setVarInit(100)

start_frame, end_frame = 0, np.inf
assert start_frame < end_frame, 'start_frame must be smaller than end_frame'

# Background model settings
warmup_learning_rate = 0.01   # learn the normal scene at startup
learning_rate = 0.0001        # very slow update during normal monitoring
freeze_learning_rate = 0.0    # freeze background after tamper is detected

# Detection settings
t_init = 150       # about 5 seconds at 30 FPS
t_fix = 600
t_set = 30
w = 700
assert w > t_fix, 'w must be larger than t_fix'

frame_no = 0
tamper_flag = 0
tamper_flag_count = 0
tamper_validation_count = 0
fixed_tamper_flag = False


In [19]:
def sobel_edge_detection(image_gray, blur_ksize=7, sobel_ksize=1, skipping_threshold=10):
    img_gaussian = cv2.GaussianBlur(
        image_gray, (blur_ksize, blur_ksize), 0
    )

    sobel_x = cv2.Sobel(
        img_gaussian, cv2.CV_64F, 1, 0, ksize=sobel_ksize
    )
    sobel_y = cv2.Sobel(
        img_gaussian, cv2.CV_64F, 0, 1, ksize=sobel_ksize
    )

    img_sobel = (np.abs(sobel_x) + np.abs(sobel_y)) / 2
    img_sobel[img_sobel < skipping_threshold] = 0
    img_sobel[img_sobel >= skipping_threshold] = 255

    return img_sobel.astype(np.uint8)


def calculate_th(e_bg):
    h, w = e_bg.shape[:2]
    edge_pixels = max(float(np.sum(e_bg) / 255), 1.0)
    edge_ratio = edge_pixels / (w * h)

    if edge_ratio < 0.052:
        return 150 / edge_pixels
    elif edge_ratio < 0.092:
        return 400 / edge_pixels
    else:
        return 1500 / edge_pixels


def calculate_quality_metrics(frame):
    # Metrics used when a scene has few or no edges, such as a wall.
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = float(np.mean(gray))
    variance = float(np.var(gray))
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    edges = cv2.Canny(gray, 50, 150)
    edge_ratio = float(np.mean(edges > 0))

    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).ravel()
    hist = hist / (hist.sum() + 1e-8)
    entropy = float(-np.sum(hist[hist > 0] * np.log2(hist[hist > 0])))

    return {
        "brightness": brightness,
        "variance": variance,
        "sharpness": sharpness,
        "edge_ratio": edge_ratio,
        "entropy": entropy,
        "gray": gray,
    }


In [20]:
aedr = 0
edr_sum = 0
bg_temp = None

aedr_list = []
edr_list = []
aedr_th_list = []
th_list = []

tamper_flag = 0
tamper_flag_count = 0
tamper_validation_count = 0
fixed_tamper_flag = False

frame_no = 0
start_frame = 0
end_frame = np.inf

t_set = 30
t_fix = 600
w = 700
sensitivity = 6.0

In [21]:
def calculate_tamper_validation_count(
    tamper_validation_count,
    w,
    edr,
    aedr,
    th,
    sensitivity=6.0,
    quality_abnormal=False
):
    edge_abnormal = edr > aedr + sensitivity * th

    if edge_abnormal or quality_abnormal:
        return min(tamper_validation_count + 1, w)

    return max(tamper_validation_count - 10, 0)


# Runtime state
aedr = 0.0
edr_sum = 0.0
bg_temp = None

aedr_list = []
edr_list = []
aedr_th_list = []
th_list = []

t_set = 30
t_fix = 600
w = 700
sensitivity = 6.0

tamper_flag = 0
tamper_flag_count = 0
tamper_validation_count = 0
fixed_tamper_flag = False

# Reference frame and quality-rule state.
reference_gray = None
anchor_reference = None
adaptive_reference = None
baseline_metrics = None
baseline_count = 0
quality_abnormal_count = 0
quality_recovery_count = 0
quality_tamper_flag = 0

frame_no = 0
start_frame = 0
end_frame = np.inf

print("开始摄像头遮挡检测")
print("正常状态：camera_status = 1")
print("遮挡状态：camera_status = 0")


while True:
    frame_no += 1
    notification = "Normal"

    ret, frame = cap.read()
    if not ret:
        print("无法读取摄像头画面")
        break

    if frame_no > start_frame:
        frame = cv2.resize(frame, (320, 240))
        quality = calculate_quality_metrics(frame)
        frame_gray = quality["gray"]

        # Establish a stable normal reference during the first 100 frames.
        if baseline_count < 100:
            if reference_gray is None:
                reference_gray = frame_gray.copy()
                baseline_metrics = {
                    key: quality[key]
                    for key in ("brightness", "variance", "sharpness", "edge_ratio", "entropy")
                }
            else:
                reference_gray = cv2.addWeighted(
                    reference_gray, 0.98, frame_gray, 0.02, 0
                )
                for key in baseline_metrics:
                    baseline_metrics[key] = (
                        0.98 * baseline_metrics[key] + 0.02 * quality[key]
                    )
            baseline_count += 1

            if baseline_count == 100:
                anchor_reference = reference_gray.copy()
                adaptive_reference = reference_gray.copy()

        cv2.imshow("frame_rgb", frame)
        cv2.moveWindow("frame_rgb", 500, 50)

        # MOG2 learns at startup, updates slowly during normal operation,
        # and freezes after the tamper state is confirmed.
        if frame_no <= 100:
            current_learning_rate = 0.01
        elif tamper_flag == 1:
            current_learning_rate = 0.0
        else:
            current_learning_rate = 0.0001

        fgmask = fgbg.apply(frame, learningRate=current_learning_rate)
        fgmask = cv2.dilate(
            fgmask,
            np.ones((5, 5), dtype=np.uint8)
        )

        # A nearby moving face can change many pixels. Ignore it unless the
        # whole image also becomes low-information or changes globally.
        motion_ratio = float(np.mean(fgmask == 255))
        quality_abnormal = False
        quality_reasons = []

        anchor_diff = 0.0
        adaptive_diff = 0.0
        changed_ratio = 0.0

        if (
            baseline_count >= 100
            and baseline_metrics is not None
            and anchor_reference is not None
            and adaptive_reference is not None
        ):
            anchor_diff = float(
                np.mean(cv2.absdiff(frame_gray, anchor_reference)) / 255.0
            )
            adaptive_diff = float(
                np.mean(cv2.absdiff(frame_gray, adaptive_reference)) / 255.0
            )
            changed_ratio = float(
                np.mean(cv2.absdiff(frame_gray, anchor_reference) > 20)
            )

            brightness_change = abs(
                quality["brightness"] - baseline_metrics["brightness"]
            ) / (baseline_metrics["brightness"] + 1.0)
            # A plain wall can already have low variance/entropy.
            # Only treat these as abnormal when they drop from a textured baseline.
            low_variance = (
                baseline_metrics["variance"] > 20.0
                and quality["variance"] < baseline_metrics["variance"] * 0.35
            )
            low_entropy = (
                baseline_metrics["entropy"] > 2.0
                and quality["entropy"] < baseline_metrics["entropy"] * 0.65
            )
            edge_drop = (
                baseline_metrics["edge_ratio"] > 0.01
                and quality["edge_ratio"] < baseline_metrics["edge_ratio"] * 0.55
            )
            hard_black_or_white = (
                quality["brightness"] < 15 or quality["brightness"] > 245
            )

            # Require a larger persistent change before calling it occlusion.
            global_scene_change = (
                (anchor_diff > 0.12 and adaptive_diff > 0.08)
                or (
                    changed_ratio > 0.25
                    and (
                        brightness_change > 0.12
                        or low_variance
                        or low_entropy
                        or edge_drop
                        or hard_black_or_white
                    )
                )
            )
            slow_occlusion = (
                changed_ratio > 0.25
                and adaptive_diff > 0.08
                and motion_ratio < 0.20
                and (
                    brightness_change > 0.12
                    or low_variance
                    or low_entropy
                    or edge_drop
                    or hard_black_or_white
                )
            )
            quality_candidate = global_scene_change or slow_occlusion

            person_moving = motion_ratio > 0.20 and not (
                low_variance or low_entropy or hard_black_or_white
            )
            quality_abnormal = quality_candidate and not person_moving

            if quality_abnormal:
                quality_abnormal_count += 1
                quality_recovery_count = 0
                quality_reasons = [
                    f"anchor_diff={anchor_diff:.2f}",
                    f"brightness_change={brightness_change:.2f}",
                ]
            else:
                quality_recovery_count += 1
                quality_abnormal_count = 0

            if quality_abnormal_count >= 15:
                quality_tamper_flag = 1
            elif quality_recovery_count >= 20:
                quality_tamper_flag = 0

        if tamper_flag == 1 and bg_temp is not None:
            bg = bg_temp.copy()
        else:
            bg = fgbg.getBackgroundImage()
            if bg is not None:
                bg_temp = bg.copy()

        if bg is None:
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
            continue

        cv2.imshow("fgmask", fgmask)
        cv2.moveWindow("fgmask", 900, 225)
        cv2.imshow("bg", bg)
        cv2.moveWindow("bg", 100, 50)

        background_gray = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)

        if frame_no > start_frame + 1:
            mask_to_exclude = (fgmask == 255)

            e_bg = sobel_edge_detection(background_gray)
            cv2.imshow("bg_sobel", e_bg)
            cv2.moveWindow("bg_sobel", 100, 400)

            e_c = sobel_edge_detection(frame_gray)
            cv2.imshow("c_sobel", e_c)
            cv2.moveWindow("c_sobel", 500, 400)

            th = calculate_th(e_bg)
            e_bg[mask_to_exclude] = 0
            e_c[mask_to_exclude] = 0
            edr = 1 - (np.sum(e_bg & e_c) / (np.sum(e_bg) + 1))

            # EMA baseline avoids indefinite drift during long runs.
            if tamper_flag == 0 and not quality_abnormal:
                if aedr == 0:
                    aedr = edr
                else:
                    aedr = 0.98 * aedr + 0.02 * edr

            if frame_no > t_init + start_frame:
                tamper_validation_count = calculate_tamper_validation_count(
                    tamper_validation_count,
                    w,
                    edr,
                    aedr,
                    th,
                    sensitivity=sensitivity,
                    quality_abnormal=(quality_tamper_flag == 1)
                )

                tamper_flag = int(tamper_validation_count > t_set)
                if tamper_flag == 1:
                    notification = "Tamper suspected"
                else:
                    notification = "Normal"
                    fixed_tamper_flag = False

                tamper_flag_count += tamper_flag

                if tamper_validation_count > t_fix:
                    notification = "Tamper attacked"
                    fixed_tamper_flag = True

            camera_status = 1 - int(bool(tamper_flag))

            print(
                "{:s} | frame no: {:d} "
                "\t edr: {:.4f} "
                "\t aedr: {:.4f} "
                "\t th: {:.4f} "
                "\t anchor diff: {:.3f} "
                "\t adaptive diff: {:.3f} "
                "\t changed ratio: {:.3f} "
                "\t quality flag: {:d} "
                "\t camera status: {:d} "
                "\t validation count: {:d} "
                "\t status: {:s}".format(
                    time.strftime("%Y-%m-%d %H:%M:%S"),
                    frame_no,
                    edr,
                    aedr,
                    th,
                    anchor_diff,
                    adaptive_diff,
                    changed_ratio,
                    quality_tamper_flag,
                    camera_status,
                    tamper_validation_count,
                    notification
                )
            )

            aedr_list.append(aedr)
            edr_list.append(edr)
            aedr_th_list.append(aedr + th)
            th_list.append(th)

            if (
                adaptive_reference is not None
                and tamper_flag == 0
                and quality_tamper_flag == 0
                and not quality_abnormal
                and motion_ratio < 0.05
            ):
                adaptive_reference = cv2.addWeighted(
                    adaptive_reference, 0.995, frame_gray, 0.005, 0
                )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    if frame_no == end_frame:
        break


cap.release()
cv2.destroyAllWindows()


开始摄像头遮挡检测
正常状态：camera_status = 1
遮挡状态：camera_status = 0
2026-10-16 15:15:22 | frame no: 2 	 edr: 0.0528 	 aedr: 0.0264 	 th: 0.0742 	 camera status: 1 	 tamper validation count: 0 	 status: Normal
2026-10-16 15:15:22 | frame no: 3 	 edr: 0.0579 	 aedr: 0.0369 	 th: 0.0742 	 camera status: 1 	 tamper validation count: 0 	 status: Normal
2026-10-16 15:15:22 | frame no: 4 	 edr: 0.0683 	 aedr: 0.0447 	 th: 0.0742 	 camera status: 1 	 tamper validation count: 0 	 status: Normal
2026-10-16 15:15:22 | frame no: 5 	 edr: 0.0801 	 aedr: 0.0518 	 th: 0.0742 	 camera status: 1 	 tamper validation count: 0 	 status: Normal
2026-10-16 15:15:22 | frame no: 6 	 edr: 0.0874 	 aedr: 0.0577 	 th: 0.0742 	 camera status: 1 	 tamper validation count: 0 	 status: Normal
2026-10-16 15:15:22 | frame no: 7 	 edr: 0.0936 	 aedr: 0.0629 	 th: 0.0742 	 camera status: 1 	 tamper validation count: 0 	 status: Normal
2026-10-16 15:15:22 | frame no: 8 	 edr: 0.1058 	 aedr: 0.0682 	 th: 0.0742 	 camera status: 1 	 t

KeyboardInterrupt: 

In [ ]:
# # eval    (turn on these code for eval)
# print('detection rate: {:.2f}'.format(tp/(tp+fn)))
# print('precision rate: {:.2f}'.format(tp/(tp+fp)))
# print('daily false alarm rate: {:.2f}'.format(fp/frame_no))

In [ ]:
# plot
import matplotlib.pyplot as plt
plt.figure(figsize=(16, 9))
plt.plot(aedr_list, 'r', label='AEDR')
plt.plot(edr_list, 'b', linestyle = ':', label='EDR')
plt.plot(aedr_th_list, 'g', linestyle = '-.', label='AEDR + TH')
plt.plot(th_list, 'brown', linestyle = '--', label='TH')
plt.ylim([0, 1])
plt.xlim([0, len(th_list)])
plt.title('If the blue line exceeds the green line, it is camera tamper attack')
plt.xlabel('Frame number + {:d}'.format(start_frame))
plt.ylabel('Edge disappearance ratio')
plt.legend()
# Display
plt.show()